In [ ]:
from z3 import *

#TODO not able even to generate sudoku with 2 empty boxes,it takes longer than 1hour

instance = ((5,3,0,0,7,0,0,0,0),
            (6,0,0,1,9,5,0,0,0),
            (0,9,8,0,0,0,0,6,0),
            (8,0,0,0,6,0,0,0,3),
            (4,0,0,8,0,3,0,0,1),
            (7,0,0,0,2,0,0,0,6),
            (0,6,0,0,0,0,2,8,0),
            (0,0,0,4,1,9,0,0,5),
            (0,0,0,0,8,0,0,7,9))

def CreateBoard(name):
    return [ [ Int(name+"_%s_%s" % (i+1, j+1)) for j in range(9) ] for i in range(9) ]

instance = CreateBoard("instance")
solved_inst = CreateBoard("solved")
solved_inst_vars=sum(solved_inst,[])
second_solution = CreateBoard("second")
second_solution_vars=sum(second_solution,[])

def IsValidSudoku(board):
    cells_c  = [ And(1 <= board[i][j], board[i][j] <= 9) for i in range(9) for j in range(9) ]
    rows_c   = [ Distinct(board[i]) for i in range(9) ]
    cols_c   = [ Distinct([ board[i][j] for i in range(9) ]) for j in range(9) ]
    sq_c     = [ Distinct([ board[3*i0 + i][3*j0 + j] for i in range(3) for j in range(3) ])    for i0 in range(3) for j0 in range(3) ]
    return And(cells_c + rows_c + cols_c + sq_c)

def IsPartOf(board1,board2):
    return And([ Implies(board1[i][j] != 0, board2[i][j] == board1[i][j])  for i in range(9) for j in range(9) ])

def IsEqual(board1,board2):
    return And([ board2[i][j] == board1[i][j] for i in range(9) for j in range(9) ])

valid_instance_c  = [ And(0 <= instance[i][j], instance[i][j] <= 9) 
             for i in range(9) for j in range(9) ]

empty_cells_instance  = Sum([ instance[i][j] == 0 for i in range(9) for j in range(9) ])



s = Solver()
s.add(Exists(solved_inst_vars,And(
                                IsPartOf(instance,solved_inst),
                                IsValidSudoku(solved_inst),
                                Not(Exists(second_solution_vars,And(
                                                            IsValidSudoku(second_solution),
                                                            IsPartOf(instance,second_solution),
                                                            Not(IsEqual(solved_inst,second_solution)))))
)))
s.add(valid_instance_c+[empty_cells_instance==2])
if s.check() == sat:
    m = s.model()
    print("instance")
    r = [ [ m.evaluate(instance[i][j]).as_long() for j in range(9) ] 
          for i in range(9) ]
    for x in r:
        print(x)

    print("solution")
    r = [ [ m.evaluate(solved_inst[i][j]) for j in range(9) ] 
          for i in range(9) ]
    for x in r:
        print(x)
else:
    print("failed to solve")



